# nmtc-screener — NMTC Feasibility Screener
## Demo: Screening Community Projects for NMTC Eligibility

This notebook demonstrates how to use nmtc-screener to assess whether
a community project qualifies for New Markets Tax Credit financing,
estimate the allocation size needed, and understand the economics.

Built for CDFIs, nonprofits, and community organizations that cannot
afford $10,000–$50,000 in consulting fees for NMTC feasibility analysis.


In [ ]:
import sys
sys.path.insert(0, '..')

# nmtc-screener builds on nmtc-calc
from nmtccalc import NMTCDeal, transaction, credits, investor, subsidy
import pandas as pd


## 1. Project 1 — Community Health Center (Chicago, IL)

A federally qualified health center serving a low-income census tract
on the south side of Chicago. Total project cost: $8MM.


In [ ]:
health_center = NMTCDeal(
    project_name="Southside Community Health Center",
    total_project_cost=8_000_000,
    nmtc_allocation=8_000_000,
    credit_price=0.83,
    leverage_loan_rate=0.045,
    qlici_a_loan_rate=0.045,
    qlici_b_loan_rate=0.010,
    cde_fee_rate=0.02,
    compliance_years=7,
    discount_rate=0.08,
    investor_name="US Bancorp CDC",
    cde_name="Chicago Development Fund",
    project_location="Chicago, IL",
)

print(health_center)
print(f"\nTotal NMTCs:      ${health_center.total_nmtcs/1e6:.2f}MM")
print(f"Investor Equity:  ${health_center.investor_equity/1e6:.2f}MM")
print(f"Leverage Loan:    ${health_center.leverage_loan/1e6:.2f}MM")


## 2. Capital Stack Structure

In [ ]:
tx = transaction.structure(health_center)
tx.summary()


## 3. 7-Year Tax Credit Schedule

In [ ]:
cr = credits.schedule(health_center)
cr.summary()


## 4. Investor Economics

In [ ]:
inv = investor.analyze(health_center)
inv.summary()


## 5. Net Subsidy to the Project

In [ ]:
sub = subsidy.analyze(health_center)
sub.summary()


## 6. Project 2 — Affordable Housing Development (Atlanta, GA)

A mixed-income housing development in an underserved census tract.
Total project cost: $15MM with partial NMTC financing.


In [ ]:
housing = NMTCDeal(
    project_name="Westside Affordable Housing",
    total_project_cost=15_000_000,
    nmtc_allocation=10_000_000,
    credit_price=0.82,
    leverage_loan_rate=0.050,
    qlici_a_loan_rate=0.050,
    qlici_b_loan_rate=0.010,
    cde_fee_rate=0.025,
    compliance_years=7,
    discount_rate=0.08,
    project_location="Atlanta, GA",
)

transaction.structure(housing).summary()
subsidy.analyze(housing).summary()


## 7. Project 3 — Small Manufacturing Facility (Detroit, MI)

A small manufacturer creating 45 jobs in a distressed community.
Total project cost: $5MM — smaller deal, tests minimum viability.


In [ ]:
manufacturing = NMTCDeal(
    project_name="Detroit Advanced Manufacturing Co.",
    total_project_cost=5_000_000,
    nmtc_allocation=5_000_000,
    credit_price=0.80,
    leverage_loan_rate=0.055,
    qlici_a_loan_rate=0.055,
    qlici_b_loan_rate=0.010,
    cde_fee_rate=0.02,
    compliance_years=7,
    discount_rate=0.09,
    project_location="Detroit, MI",
)

transaction.structure(manufacturing).summary()
subsidy.analyze(manufacturing).summary()


## 8. Compare All Three Projects

In [ ]:
deals = [health_center, housing, manufacturing]

rows = []
for deal in deals:
    sub_result = subsidy.analyze(deal)
    rows.append({
        "Project": deal.project_name,
        "Location": deal.project_location,
        "Total Cost ($MM)": f"${deal.total_project_cost/1e6:.1f}",
        "NMTC Allocation ($MM)": f"${deal.nmtc_allocation/1e6:.1f}",
        "Total NMTCs ($MM)": f"${deal.total_nmtcs/1e6:.2f}",
        "Investor Equity ($MM)": f"${deal.investor_equity/1e6:.2f}",
        "Net Subsidy ($MM)": f"${sub_result.net_subsidy/1e6:.2f}",
        "Subsidy as % of Cost": f"{sub_result.net_subsidy_pct*100:.1f}%",
        "Eff. Cost of Capital": f"{sub_result.effective_cost_of_capital*100:.2f}%",
    })

df = pd.DataFrame(rows)
print("\nNMTC Deal Comparison")
print("=" * 100)
print(df.to_string(index=False))


## 9. Credit Price Sensitivity

How does investor equity change as credit price moves from $0.75 to $0.88?


In [ ]:
credit_prices = [0.75, 0.78, 0.80, 0.82, 0.83, 0.85, 0.88]

print(f"Credit Price Sensitivity — {health_center.project_name}")
print(f"QEI: ${health_center.nmtc_allocation/1e6:.1f}MM")
print("-" * 55)
print(f"{'Credit Price':<15} {'Investor Equity':<20} {'Leverage Loan'}")
print("-" * 55)

for price in credit_prices:
    test_deal = NMTCDeal(
        project_name=health_center.project_name,
        total_project_cost=health_center.total_project_cost,
        nmtc_allocation=health_center.nmtc_allocation,
        credit_price=price,
        leverage_loan_rate=health_center.leverage_loan_rate,
        qlici_a_loan_rate=health_center.qlici_a_loan_rate,
        qlici_b_loan_rate=health_center.qlici_b_loan_rate,
        cde_fee_rate=health_center.cde_fee_rate,
    )
    print(f"  ${price:.2f}/\$1          ${test_deal.investor_equity/1e6:.2f}MM             ${test_deal.leverage_loan/1e6:.2f}MM")


## Summary

This notebook demonstrated:

1. Defining NMTC deals for three different project types
2. Computing full capital stack structure (QEI, QLICI A/B, leverage loan)
3. Generating 7-year tax credit schedules
4. Analyzing investor IRR and MOIC
5. Computing net subsidy and effective cost of capital to the borrower
6. Comparing multiple deals side by side
7. Running credit price sensitivity analysis

**The key insight:** NMTC financing delivers 20-30% of project cost as a
permanent subsidy at below-market interest rates — making otherwise
infeasible projects viable in low-income communities.

**GitHub:** https://github.com/Jaypatel1511/nmtc-screener
**PyPI:** https://pypi.org/project/nmtc-screener
**Depends on:** https://pypi.org/project/nmtc-calc
